# RUN_ON_COLAB — full neural grid on a GPU runtime

The executed notebooks in `notebooks/` run the neural models on a cross-shaped sub-grid with 3 seeds (2 CPU cores, about 3 h). This notebook extends that to the **full** design: 4 sampling intervals × 6 noise levels × 5 seeds × {N1, N2, N2-WS} × {Lorenz, LV4}, which is 720 trainings.

**What to expect from the GPU.** The states are 3- or 4-dimensional and the networks are small. Each RK4 step is a handful of tiny kernels, so a T4 does not run a single training much faster than one CPU core. What the GPU buys is 4× larger window batches for N1 (512 instead of 128) at almost no extra cost per iteration, which gives lower-variance gradients for the same number of steps. Budget about 15–25 h in total. The grid is chunked, so it can be spread over several sessions: finished models are cached on Google Drive and skipped on restart.

**Steps**
1. Runtime → Change runtime type → GPU.
2. Upload `sparse-chaos-discovery.zip` to the root of your Google Drive (*My Drive*).
3. Run all cells. Results are appended to `results/metrics_colab/` on Drive after every job.

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ZIP = "/content/drive/MyDrive/sparse-chaos-discovery.zip"   # uploaded code archive
WORK = "/content/drive/MyDrive/sparse-chaos-discovery-colab"      # persistent working copy (model cache lives here)
if not os.path.exists(os.path.join(WORK, "src")):
    os.makedirs(WORK, exist_ok=True)
    subprocess.run(["unzip", "-oq", DRIVE_ZIP, "-d", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
!pip -q install -r requirements.txt
print(sorted(os.listdir(WORK)))

In [ ]:
import torch
os.environ["SCD_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", os.environ["SCD_DEVICE"], torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
import numpy as np, pandas as pd, time, json, traceback
from src import experiment as E
from src.node import TrainConfig
OUT = os.path.join(WORK, "results", "metrics_colab"); os.makedirs(OUT, exist_ok=True)

Smoke test: one short training of each model type on the GPU (about 1–2 min). If this cell fails, stop and send me the traceback.

In [ ]:
t0 = time.time()
row, _, _ = E.run_neural("lorenz", 4/15, 0.0, 0, "N2-Lib", force=True, tag_extra="_smoke", evaluate=False)
print("N2 ok: coef_err = %.2e, exact support = %s, %.0f s" % (row["coef_err"], row["exact_support"], time.time() - t0))
t0 = time.time()
row, _, _ = E.run_neural("lorenz", 1/15, 0.01, 0, "N1-MLP", cfg=TrainConfig(schedule=[(0.1, 50), (0.25, 50)], batch=512),
                         force=True, tag_extra="_smoke", evaluate=True)
print("N1 ok: VPT_ID = %.2f, %.0f s" % (row["vpt_id"], time.time() - t0))

In [ ]:
SYSTEMS = ["lorenz", "lv4"]
METHODS = ["N2-Lib", "N2-Lib-WS", "N1-MLP"]
SEEDS = [0, 1, 2, 3, 4]
GRID = [(r, e) for r in E.DT_RATIOS for e in E.NOISES]
CHUNK = None           # e.g. slice(0, 240) to run a third of the jobs in this session
jobs = [(n, r, e, s, m) for m in METHODS for s in SEEDS for n in SYSTEMS for (r, e) in GRID]
jobs = jobs[CHUNK] if CHUNK else jobs
print(len(jobs), "jobs")

In [ ]:
log_path = os.path.join(OUT, "neural_grid_full.csv")
done = set()
if os.path.exists(log_path):
    prev = pd.read_csv(log_path)
    done = set(zip(prev.system, prev.dt_ratio.round(5), prev.noise, prev.seed, prev.method))
t0 = time.time()
for k, (n, r, e, s, m) in enumerate(jobs):
    if (n, round(r, 5), e, s, m) in done:
        continue
    cfg = TrainConfig(schedule=list(E.N1_SCHEDULE), batch=512 if os.environ["SCD_DEVICE"] == "cuda" else 128)
    try:
        row, _, _ = E.run_neural(n, r, e, s, m, cfg=cfg)
        row.pop("Xi", None)
    except Exception:
        row = dict(system=n, dt_ratio=r, noise=e, seed=s, method=m, error=traceback.format_exc()[-500:])
    pd.DataFrame([{k_: v for k_, v in row.items() if np.isscalar(v)}]).to_csv(
        log_path, mode="a", header=not os.path.exists(log_path), index=False)
    if k % 10 == 0:
        print(f"{k}/{len(jobs)}  {(time.time() - t0) / 3600:.2f} h", flush=True)

In [ ]:
# package the results to send back (metrics + cached models)
!cd "{WORK}" && zip -qr /content/drive/MyDrive/neural_grid_full_results.zip results/metrics_colab results/models
print("written to MyDrive/neural_grid_full_results.zip")